# HELIOS Neural Evaluation Harness — Google Colab GPU Environment

This notebook provides a reproducible, hardware-accelerated benchmark environment for evaluating the **HELIOS** model family (Nano 17.84M to Flagship 5.80B) on NVIDIA GPUs (T4, V100, A100, H100).

### Verification Invariants:
- **Source of Truth**: Evaluates native `HeliosForCausalLM` PyTorch checkpoints directly.
- **Deterministic**: Fixed seed `42`, greedy decoding ($T=0.0$), zero benchmark data contamination.
- **Immutable Manifests**: Generates SHA-256 verifiable JSON experiment manifests.

## 1. Environment Verification & GPU Detection

In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Clone Repository & Install Dependencies

In [ ]:
# If running directly on Colab, clone the repository
!git clone https://github.com/darknecrocities/Helios.git
%cd Helios
!pip install -q typer rich pyyaml pydantic psutil matplotlib

## 3. Checkpoint Inspection & Integrity Verification

In [ ]:
# Inspect checkpoint parameters, layers, GQA heads, and weight SHA-256
!python -m evaluation.cli inspect configs/models/nano.yaml

## 4. Single-Sample Smoke Verification
Verify all 7 benchmark task pipelines end-to-end (MMLU, GSM8K, SciQ, MedMCQA, HumanEval, HMA, BFCL).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
!python -m evaluation.cli smoke configs/models/nano.yaml --device {device}

## 5. Execute Full Standardized Benchmark Suite

In [ ]:
!python -m evaluation.cli run configs/models/nano.yaml \
    --tasks all \
    --device {device} \
    --output-dir evaluation_records \
    --seed 42

## 6. Compare Against Open-Weight Baselines & Generate Reports

In [ ]:
import glob

manifests = sorted(glob.glob("evaluation_records/manifest_*.json"))
if manifests:
    latest_manifest = manifests[-1]
    print(f"Latest manifest: {latest_manifest}")
    !python -m evaluation.cli compare {latest_manifest} --baselines-file baselines/model_registry.yaml
    !python -m evaluation.cli report {latest_manifest} --output-html report.html --output-md report.md
    print("Reports generated: report.html and report.md")